# Chemical Composition (PMDCo): RDF Generation Workflow

This notebook demonstrates the full pipeline:

1. Load a **simplified JSON input** (`simplified/example.input.json`)
2. **Transform** it to the OO-LD structure using `simplified/transform.jsonata`
3. **Convert** the OO-LD JSON to RDF (rdflib native JSON-LD 1.1 parser)
4. **Validate** the RDF graph against the SHACL shape (`shape.ttl`)

---

## Environment setup

Create and activate a virtual environment, then launch JupyterLab from within it:

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install jupyterlab
jupyter lab
```

JupyterLab will use the active venv automatically. Run the install cell below to add the required packages.

In [ ]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

In [ ]:
import sys
import json
import pathlib
import yaml
import jsonata
from importlib.metadata import version
import rdflib
import pyshacl

print("Python :", sys.executable)
print()
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

# Paths
HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # PMDCo/

---
## Step 1 — Load the simplified input

In [ ]:
simplified_path = SCHEMA / "simplified" / "example.input.json"
simplified = json.loads(simplified_path.read_text())

print(json.dumps(simplified, indent=2))

---
## Step 2 — Transform to OO-LD JSON

`simplified/transform.jsonata` is a [JSONata](https://jsonata.org) expression that
maps the simplified input to the nested OO-LD structure.
[jsonata-python](https://github.com/rayokota/jsonata-python) is the off-the-shelf
Python runtime — no custom code involved.

The same transform can also be run from the command line:
```
python -m jsonata -e ../simplified/transform.jsonata -i ../simplified/example.input.json
```

In [ ]:
expr = (SCHEMA / "simplified" / "transform.jsonata").read_text()

In [ ]:
oold_doc = jsonata.Jsonata(expr).evaluate({
    **simplified,
    "material_id": "mat_316L",
    "comp_id":     "chem_comp_316L",
})

print(json.dumps(oold_doc, indent=2))

---
## Step 3 — Convert to RDF

rdflib ≥ 7 has a native JSON-LD 1.1 parser that reads the context directly from
`schema.oold.yaml` — no preprocessing or copy-paste needed.

In [ ]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
# Optional: save to file
out_ttl = HERE / "output_316L.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

---
## Step 4 — Validate against the SHACL shape

The SHACL shape (`shape.ttl`) enforces:
- every `ChemicalComposition` has a `quality_of` link to a labelled material
- every `FractionValueSpecification` has a numeric value in \[0, 100\] and one of the three allowed unit IRIs
- every `Proportion` has exactly one `specified_by_value` and one `relational_quality_of`

> **Note:** `inference="rdfs"` is required because some shapes target superclasses
> (`PMD_0020101 Proportion`, `PMD_0020140 PortionOfSingleChemicalElement`) and
> rely on RDFS subclass reasoning to match the specific subtypes used here.

In [ ]:
shapes_graph = rdflib.Graph().parse(str(SCHEMA / "specs" / "shape.ttl"))

conforms, results_graph, results_text = pyshacl.validate(
    g,
    shacl_graph  = shapes_graph,
    inference    = "rdfs",
    serialize_report_graph = True,
)

print(f"Conforms: {conforms}")
print()
print(results_text)

---
## Recap

| Step | Tool | Input | Output |
|---|---|---|---|
| Load | —  | `simplified/example.input.json` | Python dict |
| Transform | `jsonata-python` + `simplified/transform.jsonata` | simplified dict | OO-LD dict |
| Convert to RDF | `rdflib` (JSON-LD 1.1) + `schema.oold.yaml` context | OO-LD dict | RDF graph |
| Validate | `pyshacl` + `shape.ttl` | RDF graph | Conformance report |

To use your own material, edit `simplified/example.input.json` (or point to your
own file), re-run all cells.  No code changes needed.